<!-- nav -->
[← 3 · Moving house: editing relations](03_Moving_House.ipynb) · [Home](../../README.md) · [5 · A price list: cardinality and validation →](05_A_Price_List.ipynb)

# 4 · A family tree: self-relations, cycles and reachability

**The problem.** A genealogy app. People have parents and children, some people mentor others, and some children
have a legal guardian that may be a person *or* an institution. In a typical object model, "parents" and "children"
are two arrays that must be kept in sync by hand. Cycles (a mentor mentoring their own parent) break naive
serializers, which is exactly what `JSON.stringify` does with a circular reference. And "load everyone related to
Ada" is a query you'd write yourself.

With relations:

- a relation can link two objects of the **same** schema,
- "parents" and "children" are two views of **one** relation, so they can't disagree,
- cycles and self-loops are ordinary data, and
- `Reachable.of(x)` answers "everything connected to x".

In [1]:
import { Proxies, Reachable, Schemas } from "@mbse/schemas/Framework";
import { entries } from "./toolkit.js";

const store = new Proxies.OfStore();

## One relation, two adjacencies on the same schema

`Parentage` links a `parent` and a `child`. `Person` declares *both* sides, under different accessor names. Links
are untyped, which is what makes this possible: nothing says a parent and a child must be different kinds of thing.

In [2]:
let Person = new Schemas.OfObject.Builder().name("Person").ref()
  .properties(
    (p) => p.name("name").of((t) => t.as_native(String)),
    (p) => p.name("born").of((t) => t.as_native(BigInt)), // year
  )
  .create();
const Parentage = new Schemas.OfRelation.Builder().name("Parentage")
  .links("parent", "child")
  .properties((p) => p.name("kind").of((t) => t.as_native(String))) // 'biological', 'adoptive'
  .create();
const Mentorship = new Schemas.OfRelation.Builder().name("Mentorship").links("mentor", "mentee").create();

Person = new Schemas.OfObject.Builder(Person)
  .relations(
    (adj) => adj.name("children").of(Parentage).me("parent"),
    (adj) => adj.name("parents").of(Parentage).me("child"),
    (adj) => adj.name("mentees").of(Mentorship).me("mentor"),
    (adj) => adj.name("mentors").of(Mentorship).me("mentee"),
  )
  .update();
for (const schema of [Person, Parentage, Mentorship]) {
  if (schema.validate().length > 0) throw new Error(schema.name);
  store.register(schema);
}

function names(person: Proxies.Instance, adjacency: string, link: string): string[] {
  return entries(person, adjacency).map((e) => e[link].name).sort();
}

Integers are `BigInt` in this binding, so a year is written `1815n`. JavaScript's `number` can't tell `1` from
`1.0`, and loses precision above 2^53, while the framework treats integers and floats as different types with the
same meaning in every language. Case study 5 comes back to why that matters.

## Building a family from either side

Adding a child from the parent's side and adding a parent from the child's side are the same operation. The accessor
you use only decides which link the object you're building fills:

In [3]:
const ada = store.Person().name("Ada").born(1815n).create();

const byron = store.Person()
  .name("Byron")
  .born(1836n)
  .parents((x: any) => x.parent(ada).kind("biological")) // added from the child's side
  .create();
store.Person(ada)
  .children((x: any) => x.child((c: any) => c.name("Annabella").born(1837n)).kind("biological")) // from the parent's side, inline
  .update();

console.log("Ada's children:  ", names(ada, "children", "child"));
console.log("Byron's parents: ", names(byron, "parents", "parent"));

Ada's children:   [ "Annabella", "Byron" ]


Byron's parents:  [ "Ada" ]


There's no "sync the inverse array" step to forget: `children` and `parents` are two windows onto the same entries.

## Several relations between the same pair, and more than one owner

A person can have two parents (two entries), and the same two people can be related in two different ways (two
relations). None of these interfere:

In [4]:
const william = store.Person().name("William").born(1805n).create();
store.Person(byron).parents((x: any) => x.parent(william).kind("biological")).update();
store.Person(ada).mentees((x: any) => x.mentee(byron)).update();

console.log("Byron's parents:", names(byron, "parents", "parent"));
console.log("Byron's mentors:", names(byron, "mentors", "mentor"));

Byron's parents: [ "Ada", "William" ]


Byron's mentors: [ "Ada" ]


## Cycles and self-loops are just data

Byron mentors his mother (a cycle through two relations), and William mentors himself (a self-loop):

In [5]:
store.Person(byron).mentees((x: any) => x.mentee(ada)).update();
store.Person(william).mentees((x: any) => x.mentee(william)).update();

console.log("William mentors:", names(william, "mentees", "mentee"), "| mentored by:", names(william, "mentors", "mentor"));

William mentors: [ "William" ] | mentored by: [ "William" ]


## Everything connected: `Reachable.of`

`Reachable.of(root)` returns the root and every object reachable from it through *any* adjacency, in the order they
were first reached (breadth-first). Cycles are visited once. It follows links in both directions, because an entry
is visible from every end. So it finds the whole *connected component*, not just "descendants":

In [6]:
console.log(Reachable.of(byron).map((p) => (p as Proxies.Instance).name));

[ "Byron", "Ada", "William", "Annabella" ]


In [7]:
const loner = store.Person().name("Eve").create();
console.log(Reachable.of(loner).map((p) => (p as Proxies.Instance).name));

[ "Eve" ]


**Why does this matter?** It's exactly the set of objects you must save together so that every link in them still
points at something. Case study 6 uses it to save and load whole graphs. It doesn't recurse, so a 100,000-person
chain won't overflow the call stack.

## When a link can be filled by different kinds of thing

Guardianship: a child's guardian may be a `Person` or an `Institution`. Both declare an adjacency filling
`guardian`. That's perfectly legal: links are untyped, and a link can be filled by any object whose schema says it
can fill it.

In [8]:
let Institution = new Schemas.OfObject.Builder().name("Institution").ref().properties((p) => p.name("name").of((t) => t.as_native(String))).create();
const Guardianship = new Schemas.OfRelation.Builder().name("Guardianship").links("guardian", "ward").create();

Institution = new Schemas.OfObject.Builder(Institution).relations((adj) => adj.name("wards").of(Guardianship).me("guardian")).update();
Person = new Schemas.OfObject.Builder(Person)
  .relations(
    (adj) => adj.name("wards").of(Guardianship).me("guardian"),
    (adj) => adj.name("guardians").of(Guardianship).me("ward"),
  )
  .update();
store.register(Institution);
store.register(Guardianship);

Linking existing objects works for either kind:

In [9]:
const court = store.Institution().name("Court of Chancery").create();
store.Person(byron).guardians((x: any) => x.guardian(court)).update();
store.Person(byron).guardians((x: any) => x.guardian(william)).update();
console.log(entries(byron, "guardians").map((e) => [e.guardian.schema_name(), e.guardian.name]).sort());

[ [ "Institution", "Court of Chancery" ], [ "Person", "William" ] ]


But building the guardian *inline* is refused. With two schemas able to fill `guardian`, the framework would have to
guess which one you meant, and it won't:

In [10]:
try {
  store.Person(byron).guardians((x: any) => x.guardian((g: any) => g.name("?"))).update();
} catch (error) {
  console.log(String(error));
}

TypeError: expected exactly one object schema filling link 'guardian', found ['Person', 'Institution']


Build it first with the builder you mean (`store.Institution()...create()`), then link it. Refusing to guess is a
recurring choice in this framework. The alternative, silently picking the first match, would work until someone
registered a new schema, and then change meaning.

`schema_name()` tells you which kind an object is. Every instance carries it, and case study 6 shows it travelling
with references on the wire.

## What you learned

- A relation can link objects of one schema. Each side is an adjacency with its own accessor name.
- Inverse views (`parents`/`children`) are the same entries, so they're always consistent.
- Multiple owners, several relations per pair, cycles and self-loops need no special handling.
- `Reachable.of(x)` lists x's connected component, root first, in first-reference order.
- A link filled by several schemas accepts any of them. Inline creation needs exactly one.

**Next:** case study 5 adds rules. It covers *how many* entries a relation may have (a price per currency, one owner
per pet) and how to check data against its schema when you decide it's time.

<!-- nav -->
[← 3 · Moving house: editing relations](03_Moving_House.ipynb) · [Home](../../README.md) · [5 · A price list: cardinality and validation →](05_A_Price_List.ipynb)